In [7]:
import geopandas as gpd
g = gpd.read_file('./data/export.geojson')
g.head()

,id,@id,access,addr:city,addr:country,addr:floor,addr:housenumber,addr:postcode,addr:state,addr:street,...,tailor:alteration_service,takeaway,toilets,toilets:access,tourism,twitter,website,wheelchair,@geometry,geometry
0,way/241829917,way/241829917,NaN,NaN,NaN,NaN,586,11218,NaN,Coney Island Avenue,...,NaN,NaN,NaN,NaN,NaN,NaN,https://www.cleanritecenter.com/stores/164cr/,NaN,center,POINT (-73.97023 40.64288)
1,way/247462746,way/247462746,NaN,NaN,NaN,NaN,2540,11208,NaN,Linden Boulevard,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,center,POINT (-73.87008 40.66686)
2,way/247638240,way/247638240,NaN,NaN,NaN,NaN,88-12,11417,NaN,Liberty Avenue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,center,POINT (-73.85004 40.67968)
3,way/247860012,way/247860012,NaN,Jamaica,NaN,NaN,184-17,11432,NY,Hillside Avenue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,center,POINT (-73.77744 40.71442)
4,way/247923749,way/247923749,NaN,Brooklyn,NaN,NaN,1755,11230,NY,Coney Island Avenue,...,NaN,NaN,NaN,NaN,NaN,NaN,https://atlantissuperwash.com,NaN,center,POINT (-73.96278 40.61459)


In [9]:
g['lng'] = g.geometry.x
g['lat'] = g.geometry.y
 
out = g[['name','lat','lng']]
out.head()

,name,lat,lng
0,Clean Rite Center,40.642885,-73.970234
1,Laundryland,40.666863,-73.870085
2,NaN,40.679685,-73.850040
3,Ping Laundromat,40.714422,-73.777439
4,Atlantis Super Wash Laundromat,40.614593,-73.962784


In [13]:
import h3

out['h3_res7'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 7), axis=1)
out['h3_res8'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 8), axis=1)
out['h3_res9'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 9), axis=1)

final = out[['name', 'lat', 'lng', 'h3_res7', 'h3_res8', 'h3_res9']]
final

,name,lat,lng,h3_res7,h3_res8,h3_res9
0,Clean Rite Center,40.642885,-73.970234,872a10775ffffff,882a107753fffff,892a107752bffff
1,Laundryland,40.666863,-73.870085,872a100c8ffffff,882a100c87fffff,892a100c877ffff
2,NaN,40.679685,-73.850040,872a100ccffffff,882a100cc3fffff,892a100cc2fffff
3,Ping Laundromat,40.714422,-73.777439,872a100edffffff,882a100ed1fffff,892a100edc7ffff
4,Atlantis Super Wash Laundromat,40.614593,-73.962784,872a10744ffffff,882a107441fffff,892a1074417ffff
...,...,...,...,...,...,...
1040,Sun Laundry & Cleaners,40.777156,-73.951898,872a10089ffffff,882a10089bfffff,892a10089abffff
1041,Junction Laundromat,40.756192,-73.873139,872a100f1ffffff,882a100c4dfffff,892a100c4dbffff
1042,NaN,40.695763,-73.839347,872a100ccffffff,882a100cc9fffff,892a100cc97ffff
1043,Ka Ka Laundromat,40.639406,-74.012445,872a10773ffffff,882a10773dfffff,892a10773d7ffff


In [15]:
final.to_csv("./data/laundry_sample_prelim.csv", index=False, encoding='utf-8')

Making the ETL code
-------------------

In [1]:
import geopandas as gpd
import h3

g = gpd.read_file('./data/export.geojson')

g['lng'] = g.geometry.x
g['lat'] = g.geometry.y
 
out = g[['name','lat','lng']]

out['h3_res7'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 7), axis=1)
out['h3_res8'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 8), axis=1)
out['h3_res9'] = out.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 9), axis=1)

final = out[['name', 'lat', 'lng', 'h3_res7', 'h3_res8', 'h3_res9']]
final

,name,lat,lng,h3_res7,h3_res8,h3_res9
0,Clean Rite Center,40.642885,-73.970234,872a10775ffffff,882a107753fffff,892a107752bffff
1,Laundryland,40.666863,-73.870085,872a100c8ffffff,882a100c87fffff,892a100c877ffff
2,NaN,40.679685,-73.850040,872a100ccffffff,882a100cc3fffff,892a100cc2fffff
3,Ping Laundromat,40.714422,-73.777439,872a100edffffff,882a100ed1fffff,892a100edc7ffff
4,Atlantis Super Wash Laundromat,40.614593,-73.962784,872a10744ffffff,882a107441fffff,892a1074417ffff
...,...,...,...,...,...,...
1040,Sun Laundry & Cleaners,40.777156,-73.951898,872a10089ffffff,882a10089bfffff,892a10089abffff
1041,Junction Laundromat,40.756192,-73.873139,872a100f1ffffff,882a100c4dfffff,892a100c4dbffff
1042,NaN,40.695763,-73.839347,872a100ccffffff,882a100cc9fffff,892a100cc97ffff
1043,Ka Ka Laundromat,40.639406,-74.012445,872a10773ffffff,882a10773dfffff,892a10773d7ffff


In [5]:
import requests
import pandas as pd

# The Overpass API endpoint
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# NYC Area ID is 3600175905 (found by searching NYC in Overpass Turbo)
# We add 3600000000 to the OSM Relation ID to get the Overpass Area ID
overpass_query = """
[out:json][timeout:25];
area(3600175905)->.searchArea;
(
  node["shop"="laundry"](area.searchArea);
  way["shop"="laundry"](area.searchArea);
  node["amenity"="washing-machine"](area.searchArea);
);
out center;
"""

def fetch_laundry_data():
    response = requests.post(OVERPASS_URL, data={'data': overpass_query})
    
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        
        # Extract lat/lng/tags into a flat list
        processed_data = []
        for e in elements:
            # Ways have 'center', nodes have 'lat/lon'
            lat = e.get('lat') or e.get('center', {}).get('lat')
            lon = e.get('lon') or e.get('center', {}).get('lon')
            
            processed_data.append({
                'osm_id': e['id'],
                'name': e.get('tags', {}).get('name', 'Unknown'),
                'lat': lat,
                'lng': lon,
                'amenity_type': 'laundry'
            })
            
        return pd.DataFrame(processed_data)
    else:
        print(f"Error: {response.status_code}")
        return None

df = fetch_laundry_data()
df

,osm_id,name,lat,lng,amenity_type
0,419368026,Bubbly Laundromat,40.690273,-73.995101,laundry
1,937351560,Country Club Laundromat & Dry Cleaners,40.846290,-73.819507,laundry
2,1177271154,N & K Express Laundromat,40.673701,-73.953075,laundry
3,1581645056,Clean Laundry,40.746448,-73.973691,laundry
4,1708719691,Laudromat,40.769999,-73.912397,laundry
...,...,...,...,...,...
1040,468226510,Laundromat Dry Cleaners,40.739014,-73.809711,laundry
1041,510873280,Laundry Express,40.649772,-73.963477,laundry
1042,1084910229,Farmers Laundromat,40.696878,-73.762127,laundry
1043,1188444776,Clean Rite Center,40.676788,-73.924260,laundry


In [1]:
import requests
import pandas as pd
import h3

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

overpass_query = """
[out:json][timeout:25];
area(3600175905)->.searchArea;
(
  node["shop"="laundry"](area.searchArea);
  way["shop"="laundry"](area.searchArea);
  node["amenity"="washing-machine"](area.searchArea);
);
out center;
"""

def fetch_and_index_laundry():
    response = requests.post(OVERPASS_URL, data={'data': overpass_query})
    
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        
        processed_data = []
        for e in elements:
            lat = e.get('lat') or e.get('center', {}).get('lat')
            lon = e.get('lon') or e.get('center', {}).get('lon')
            
            processed_data.append({
                'osm_id': e['id'],
                'name': e.get('tags', {}).get('name', 'Unknown'),
                'lat': lat,
                'lng': lon,
                'amenity_type': 'laundry'
            })
            
        df = pd.DataFrame(processed_data)
    
        df['h3_res7'] = df.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 7), axis=1)
        df['h3_res8'] = df.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 8), axis=1)
        df['h3_res9'] = df.apply(lambda x: h3.latlng_to_cell(x.lat, x.lng, 9), axis=1)

        final = df[['osm_id', 'name', 'amenity_type', 'lat', 'lng', 'h3_res7', 'h3_res8', 'h3_res9']]
        return final
    else:
        print(f"Error: {response.status_code}")
        return None

final_df = fetch_and_index_laundry()
# final_df.to_csv("./data/laundry_etl.csv", index=False, encoding='utf-8')

In [3]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv

load_dotenv() # This loads the variables from .env

URL = os.environ.get("SUPABASE_URL")
KEY = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(URL, KEY)

def push_to_supabase(df):
    # Convert DataFrame to a list of dictionaries (which Supabase expects)
    # .to_dict('records') turns each row into a JSON object
    data = df.to_dict(orient='records')
    
    # 2. The Upsert Command
    # 'on_conflict' tells Supabase to use 'osm_id' to check for duplicates
    try:
        response = supabase.table("amenities").upsert(
            data, 
            on_conflict="osm_id"
        ).execute()
        
        print(f"Successfully pushed {len(data)} rows to Supabase!")
        return response
    except Exception as e:
        print(f"Error pushing to Supabase: {e}")

# Run it!
push_to_supabase(final_df)

final_df

Successfully pushed 1045 rows to Supabase!


,osm_id,name,amenity_type,lat,lng,h3_res7,h3_res8,h3_res9
0,419368026,Bubbly Laundromat,laundry,40.690273,-73.995101,872a10729ffffff,882a107299fffff,892a107298bffff
1,937351560,Country Club Laundromat & Dry Cleaners,laundry,40.846290,-73.819507,872a1001cffffff,882a1001c9fffff,892a1001c9bffff
2,1177271154,N & K Express Laundromat,laundry,40.673701,-73.953075,872a100dbffffff,882a100dbdfffff,892a100dbc7ffff
3,1581645056,Clean Laundry,laundry,40.746448,-73.973691,872a100d2ffffff,882a100d05fffff,892a100d297ffff
4,1708719691,Laudromat,laundry,40.769999,-73.912397,872a100f3ffffff,882a100f3dfffff,892a100f3d7ffff
...,...,...,...,...,...,...,...,...
1040,468226510,Laundromat Dry Cleaners,laundry,40.739014,-73.809711,872a100e1ffffff,882a100e13fffff,892a100e12bffff
1041,510873280,Laundry Express,laundry,40.649772,-73.963477,872a10775ffffff,882a107751fffff,892a1077517ffff
1042,1084910229,Farmers Laundromat,laundry,40.696878,-73.762127,872a1005affffff,882a1005adfffff,892a1005ac7ffff
1043,1188444776,Clean Rite Center,laundry,40.676788,-73.924260,872a100d9ffffff,882a100d9dfffff,892a100d9d7ffff
